# Agentic AI Research Agent
**Flow:** User Query → Query Rewrite → Tavily Web Search → LLM Synthesizer → Critic → (retry?) → **Human Approval** → (approved → Done) | (rejected → Query Rewrite with feedback)

In [ ]:
!pip install -q langchain langgraph langchain-openai langchain-tavily tavily-python python-dotenv

In [ ]:
# ── CELL 1: Imports & API Keys ──────────────────────────────────────────────
# Store your keys in a .env file (never hardcode them!):
#   TAVILY_API_KEY=tvly-...
#   GROQ_API_KEY=gsk_...

import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END

load_dotenv()  # loads from .env file automatically

# Verify keys loaded
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not found in .env"
assert os.getenv("GROQ_API_KEY"),   "GROQ_API_KEY not found in .env"

print("✅ API keys loaded successfully.")

In [ ]:
# ── CELL 2: State ────────────────────────────────────────────────────────────
class ResearchState(TypedDict):
    query:          str
    web_results:    str
    search_query:   str
    final_answer:   str
    needs_retry:    bool   # critic loop flag
    retry_count:    int    # capped at 1 for critic loop
    human_approved: bool   # set by human_approval_node
    human_feedback: str    # NEW — rejection reason from human

In [ ]:
# ── CELL 3: Tools & LLM ──────────────────────────────────────────────────────

# Tavily — returns top-5 web results as structured dicts
tavily = TavilySearch(max_results=5)

# Groq LLaMA via OpenAI-compatible endpoint (free tier)
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
# ── CELL 4: Node 1 — Query Rewrite ───────────────────────────────────────────
# If human_feedback exists (i.e. human rejected previous answer),
# it's included so the rewrite is smarter — not just the same query again.

REWRITE_PROMPT = """\
You rewrite user questions into a short, high-signal web search query.
Return ONLY the search query text. No explanation, no quotes.
"""

def query_rewrite_node(state: ResearchState) -> dict:
    print("[1/4] Rewriting query for search...")

    # If human rejected and gave feedback, incorporate it
    feedback = state.get("human_feedback", "").strip()
    if feedback:
        user_content = (
            f"Original question: {state['query']}\n"
            f"Previous answer was rejected. Human feedback: {feedback}\n"
            f"Rewrite the search query to address this feedback."
        )
        print(f"    Using human feedback to improve query: '{feedback}'")
    else:
        user_content = state["query"]

    messages = [
        SystemMessage(content=REWRITE_PROMPT),
        HumanMessage(content=user_content)
    ]
    response = llm.invoke(messages)
    rewritten = response.content.strip()

    print(f"    Search query: {rewritten}")
    return {"search_query": rewritten}

In [ ]:
# ── CELL 5: Node 2 — Web Search ──────────────────────────────────────────────

def web_search_node(state: ResearchState) -> dict:
    print("[2/4] Searching the web...")

    raw = tavily.invoke({"query": state["search_query"]})
    raw = raw["results"]

    formatted_results = []
    for i, result in enumerate(raw, start=1):
        title   = result.get("title",   "No title")
        url     = result.get("url",     "")
        content = result.get("content", "").strip()

        formatted_results.append(
            f"[{i}] {title}\n"
            f"    URL: {url}\n"
            f"    {content}"
        )

    web_results_text = "\n\n".join(formatted_results)
    print(f"    Retrieved {len(raw)} sources.")

    return {"web_results": web_results_text}

In [ ]:
# ── CELL 6: Node 3 — LLM Synthesizer ─────────────────────────────────────────

SYSTEM_PROMPT = """\
You are an expert research assistant.

You will be given:
  1. A user research query.
  2. Five web search results (title, URL, snippet).

Your job:
  - Synthesize the web results with your own knowledge.
  - Write a clear, well-structured, comprehensive answer.
  - Cite sources inline using [1], [2], etc. where relevant.
  - Highlight any conflicting information across sources.
  - End with a "Sources" section listing all URLs.
  - Be factual. Do not hallucinate beyond the provided data.
"""

def synthesizer_node(state: ResearchState) -> dict:
    print("[3/4] Synthesizing answer...")

    user_prompt = (
        f"Research Query: {state['query']}\n\n"
        f"Web Search Results:\n\n{state['web_results']}"
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]

    response = llm.invoke(messages)
    return {"final_answer": response.content}

In [ ]:
# ── CELL 7: Node 4 — Critic ──────────────────────────────────────────────────

CRITIC_PROMPT = """\
You judge whether a research answer is well supported by the given sources.
Reply with exactly one word: "yes" if well supported,
"no" if it is thin, vague, or barely uses the sources.
"""

def critic_node(state: ResearchState) -> dict:
    print("[4/4] Checking answer quality...")

    messages = [
        SystemMessage(content=CRITIC_PROMPT),
        HumanMessage(content=(
            f"Sources:\n{state['web_results']}\n\n"
            f"Answer:\n{state['final_answer']}"
        ))
    ]

    verdict = llm.invoke(messages).content.strip().lower()
    needs_retry = verdict.startswith("no") and state["retry_count"] < 1

    print(f"    Verdict: {verdict} | retrying: {needs_retry}")

    return {
        "needs_retry": needs_retry,
        "retry_count": state["retry_count"] + 1
    }

In [ ]:
# ── CELL 8: Node 5 — Human Approval ──────────────────────────────────────────
# Uses interrupt() to pause graph and wait for human input.
# If human types 'ok' → approved → END
# If human types anything else → treated as feedback → loops back to query_rewrite

def human_approval_node(state: ResearchState) -> dict:
    from langgraph.types import interrupt

    print("\n" + "─" * 60)
    print(" HUMAN APPROVAL REQUIRED")
    print("─" * 60)
    print(state["final_answer"][:600], "...\n" if len(state["final_answer"]) > 600 else "\n")

    # Pause here — resume value becomes user_input
    user_input = interrupt("Type 'ok' to approve, or give feedback to improve the answer: ")

    approved = user_input.strip().lower() == "ok"

    if approved:
        print("✅ Approved!")
        return {
            "human_approved": True,
            "human_feedback": ""   # clear any previous feedback
        }
    else:
        feedback = user_input.strip()
        print(f"🔄 Rejected. Feedback: '{feedback}'. Retrying with improved query...")
        return {
            "human_approved": False,
            "human_feedback": feedback,
            "retry_count":    0       # reset critic retry counter for fresh attempt
        }

In [ ]:
# ── CELL 9: Build Graph ───────────────────────────────────────────────────────
#
#   START → query_rewrite → web_search → synthesizer → critic
#               ↑                                         |
#               └──── needs_retry=True ───────────────────┘
#                              (else)
#                                ↓
#                        human_approval   ← graph PAUSES here
#                        /           \
#                  approved         rejected (with feedback)
#                     ↓                  ↓
#                    END          query_rewrite  (smarter retry)

from langgraph.checkpoint.memory import MemorySaver

def route_after_critic(state: ResearchState):
    return "web_search" if state["needs_retry"] else "human_approval"

def route_after_human(state: ResearchState):
    # NEW — approved goes to END, rejected loops back to query_rewrite
    return END if state["human_approved"] else "query_rewrite"


workflow = StateGraph(ResearchState)

workflow.add_node("query_rewrite",  query_rewrite_node)
workflow.add_node("web_search",     web_search_node)
workflow.add_node("synthesizer",    synthesizer_node)
workflow.add_node("critic",         critic_node)
workflow.add_node("human_approval", human_approval_node)

workflow.add_edge(START,           "query_rewrite")
workflow.add_edge("query_rewrite", "web_search")
workflow.add_edge("web_search",    "synthesizer")
workflow.add_edge("synthesizer",   "critic")

workflow.add_conditional_edges(
    "critic",
    route_after_critic,
    {"web_search": "web_search", "human_approval": "human_approval"}
)

# NEW conditional edge after human approval
workflow.add_conditional_edges(
    "human_approval",
    route_after_human,
    {END: END, "query_rewrite": "query_rewrite"}
)

memory = MemorySaver()
agent  = workflow.compile(checkpointer=memory)

print("✅ Graph compiled successfully.")

In [ ]:
# ── CELL 10: Run Function ─────────────────────────────────────────────────────
# Handles the multi-phase execution:
#   Phase 1 — runs until graph pauses at human_approval
#   Phase 2+ — resumes with human input; loops if rejected until approved

def research(query: str):
    import uuid
    thread = {"configurable": {"thread_id": str(uuid.uuid4())}}

    print(f"\n{'='*60}")
    print(f" QUERY: {query}")
    print(f"{'='*60}\n")

    # ── Phase 1: Run until first interrupt ───────────────────────────────────
    state = agent.invoke(
        {
            "query":          query,
            "search_query":   "",
            "web_results":    "",
            "final_answer":   "",
            "needs_retry":    False,
            "retry_count":    0,
            "human_approved": False,
            "human_feedback": "",
        },
        config=thread
    )

    # ── Phase 2+: Human review loop ───────────────────────────────────────────
    # Keeps looping until human types 'ok' (approved)
    while not state.get("human_approved", False):

        # Show the current draft answer
        print(f"\n{'─'*60}")
        print(" DRAFT ANSWER")
        print(f"{'─'*60}\n")
        print(state["final_answer"])

        # Get human input
        print(f"\n{'─'*60}")
        human_input = input("Type 'ok' to approve, or give feedback to improve: ")

        # Resume graph with human's input string
        state = agent.invoke(
            {"resume": human_input},  # LangGraph passes this as interrupt() return value
            config=thread
        )

    # ── Final approved answer ─────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(" ✅ FINAL APPROVED ANSWER")
    print(f"{'='*60}\n")
    print(state["final_answer"])
    print(f"\n{'='*60}\n")

    return state

In [ ]:
# ── CELL 11: Test It ──────────────────────────────────────────────────────────

research("What are the latest developments in quantum computing in 2025?")

In [ ]:
# ── CELL 12: Try Your Own Query ───────────────────────────────────────────────

research("Impact of AI on software engineering jobs")